# TP2 : Training an MLP on MNIST (Keras)
**ESI-SBA — École Supérieure en Informatique 08-Mai-1945**  
**Deep Learning — Dr. N. DIF**

---

## Objectives
1. Load and preprocess the MNIST dataset
2. Split into Train / Validation sets
3. Design an MLP architecture
4. Compare SGD variants (Online, Mini-Batch, Batch)
5. Study the effect of Decay and Momentum
6. Compare SGD, Adam, and RMSprop optimizers
7. Export, import, and test the best model
8. Summarize results with comparative tables and plots

---
## 0. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

---
## 1. Load and Preprocess the MNIST Dataset

MNIST contains 70,000 grayscale images of handwritten digits (0–9), each of size 28×28 pixels.

**Preprocessing steps:**
- **Normalization**: Pixel values divided by 255 → range [0, 1]
- **Flattening**: 28×28 images reshaped to vectors of size 784
- **One-Hot Encoding**: Labels converted to 10-class binary vectors

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

# Normalize
X_train_full = X_train_full.astype('float32') / 255.0
X_test       = X_test.astype('float32') / 255.0

# Flatten 28x28 → 784
X_train_full = X_train_full.reshape(-1, 784)
X_test       = X_test.reshape(-1, 784)

# One-hot encode labels
y_train_full = to_categorical(y_train_full, 10)
y_test       = to_categorical(y_test, 10)

print(f'Training set : {X_train_full.shape}  Labels: {y_train_full.shape}')
print(f'Test set     : {X_test.shape}  Labels: {y_test.shape}')

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    img = X_train_full[i].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{np.argmax(y_train_full[i])}', fontsize=8)
    ax.axis('off')
plt.suptitle('Sample MNIST images (after normalization)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Train / Validation Split

We split the 60,000 training samples into **48,000 train** and **12,000 validation** (80/20 ratio).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42
)

print(f'Train      : {X_train.shape}')
print(f'Validation : {X_val.shape}')
print(f'Test       : {X_test.shape}')

---
## 3. MLP Architecture

```
Input (784)  →  Dense(128, ReLU)  →  Dense(64, ReLU)  →  Dense(10, Softmax)
```

| Layer | Units | Activation | Parameters |
|-------|-------|------------|------------|
| Input | 784   | —          | —          |
| Hidden 1 | 128 | ReLU    | 100,480    |
| Hidden 2 | 64  | ReLU    | 8,256      |
| Output   | 10  | Softmax | 650        |
| **Total** | — | —        | **109,386** |

**Loss function:** Categorical Cross-Entropy

In [ ]:
def build_mlp():
    model = Sequential([
        Input(shape=(784,)),
        Dense(128, activation='relu'),
        Dense(64,  activation='relu'),
        Dense(10,  activation='softmax')
    ])
    return model

# Display architecture
demo = build_mlp()
demo.summary()

---
## 4. SGD Variants: Online, Mini-Batch, and Batch

We compare three gradient descent strategies over **50 epochs** with **lr = 0.01**:

| Strategy       | Batch Size | Updates/Epoch |
|----------------|------------|---------------|
| Online SGD     | 1          | 48,000        |
| Mini-Batch SGD | 64         | 750           |
| Batch GD       | Full set   | 1             |

In [ ]:
# ── 4.1 Online SGD (batch=1) ──────────────────────────────────────────────────
print('Training Online SGD (batch=1) — this may take a while...')
model_sgd = build_mlp()
model_sgd.compile(optimizer=SGD(learning_rate=0.01),
                  loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
history_sgd = model_sgd.fit(X_train, y_train, validation_data=(X_val, y_val),
                             epochs=50, batch_size=1, verbose=1)
time_sgd = time.time() - t0
print(f'\n✅ Online SGD done — {time_sgd/60:.1f} min')

In [ ]:
# ── 4.2 Mini-Batch SGD (batch=64) ────────────────────────────────────────────
print('Training Mini-Batch SGD (batch=64)...')
model_mini = build_mlp()
model_mini.compile(optimizer=SGD(learning_rate=0.01),
                   loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
history_mini = model_mini.fit(X_train, y_train, validation_data=(X_val, y_val),
                               epochs=50, batch_size=64, verbose=1)
time_mini = time.time() - t0
print(f'\n✅ Mini-Batch SGD done — {time_mini:.1f} s')

In [ ]:
# ── 4.3 Batch Gradient Descent (full dataset) ─────────────────────────────────
print('Training Batch GD (full dataset)...')
model_batch = build_mlp()
model_batch.compile(optimizer=SGD(learning_rate=0.01),
                    loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
history_batch = model_batch.fit(X_train, y_train, validation_data=(X_val, y_val),
                                 epochs=50, batch_size=len(X_train), verbose=1)
time_batch = time.time() - t0
print(f'\n✅ Batch GD done — {time_batch:.1f} s')

In [ ]:
# ── Section 4 Plots ───────────────────────────────────────────────────────────
def plot_loss_acc(history, title, ax_loss, ax_acc):
    ax_loss.plot(history.history['loss'],     label='Train Loss',      color='steelblue')
    ax_loss.plot(history.history['val_loss'], label='Val Loss',        color='orange', linestyle='--')
    ax_loss.set_title(f'{title} — Loss'); ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)

    ax_acc.plot(history.history['accuracy'],     label='Train Acc',   color='steelblue')
    ax_acc.plot(history.history['val_accuracy'], label='Val Acc',     color='orange', linestyle='--')
    ax_acc.set_title(f'{title} — Accuracy'); ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(); ax_acc.grid(alpha=0.3)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
plot_loss_acc(history_sgd,   'Online SGD (bs=1)',      axes[0,0], axes[0,1])
plot_loss_acc(history_mini,  'Mini-Batch SGD (bs=64)', axes[1,0], axes[1,1])
plot_loss_acc(history_batch, 'Batch GD (full set)',    axes[2,0], axes[2,1])
plt.suptitle('Section 4 — SGD Strategies: Loss & Accuracy Curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('section4_sgd_strategies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 4 — Overlay Comparison ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for h, lbl, c in [(history_sgd, 'Online SGD', 'royalblue'),
                  (history_mini,'Mini-Batch SGD','darkorange'),
                  (history_batch,'Batch GD','green')]:
    ax1.plot(h.history['val_loss'],     label=lbl, color=c)
    ax2.plot(h.history['val_accuracy'], label=lbl, color=c)

ax1.set_title('Validation Loss — SGD Strategies'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Validation Accuracy — SGD Strategies'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 4 — Overlay: All SGD Strategies (Validation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('section4_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 4 — Training Time Bar Chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
names  = ['Online SGD\n(bs=1)', 'Mini-Batch SGD\n(bs=64)', 'Batch GD\n(full)']
times  = [time_sgd, time_mini, time_batch]
colors = ['royalblue', 'darkorange', 'green']
bars   = ax.bar(names, times, color=colors, edgecolor='black', width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{t:.0f}s', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Training Time (seconds)'); ax.set_title('Training Time Comparison — SGD Strategies')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig('section4_time.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print('\n── Section 4 Summary ──────────────────────────────────────────────')
print(f'{"Strategy":<25} {"Train Acc":>10} {"Val Acc":>10} {"Time":>10}')
print('-'*60)
for name, hist, t in [("Online SGD (bs=1)", history_sgd, time_sgd),
                       ("Mini-Batch SGD (bs=64)", history_mini, time_mini),
                       ("Batch GD (full)", history_batch, time_batch)]:
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    print(f'{name:<25} {ta:>10.4f} {va:>10.4f} {t:>9.1f}s')

### 📝 Remarks — Section 4

| Strategy | Final Train Acc | Final Val Acc | Training Time | Convergence Speed |
|---|---|---|---|---|
| Online SGD (bs=1) | ~99.6% | ~97.8% | Very slow (~90 min) | Fast per epoch but noisy |
| Mini-Batch SGD (bs=64) | ~98.7% | ~97.3% | Fast (~2 min) | Smooth, steady |
| Batch GD (full) | ~42.8% | ~43.3% | Medium (~15s) | Very slow, barely learning |

**Key observations:**
- **Online SGD** achieves high accuracy due to noisy updates that act as regularization, but is impractically slow.
- **Mini-Batch SGD (bs=64)** offers the best trade-off: good accuracy, efficient GPU utilization, and stable convergence.
- **Batch GD** converges extremely slowly at lr=0.01; 50 epochs is insufficient with a single update per epoch. It needs a much higher learning rate or more epochs.

---
## 5. Effect of Decay and Momentum

We compare **three variants** of mini-batch SGD (bs=64) over 50 epochs:

| Model | LR | Decay | Momentum |
|---|---|---|---|
| Mini-Batch SGD | 0.01 | — | — |
| SGD + Decay | 0.01 | 1e-6 | — |
| SGD + Momentum | 0.01 | — | 0.9 |

In [ ]:
# ── 5.1 Mini-Batch SGD with Decay ─────────────────────────────────────────────
from tensorflow.keras.optimizers.schedules import ExponentialDecay

lr_schedule_decay = ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=750,   # one epoch worth of steps
    decay_rate=0.9996  # equivalent to decay ≈ 1e-6
)

model_decay = build_mlp()
model_decay.compile(optimizer=SGD(learning_rate=lr_schedule_decay),
                    loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
history_decay = model_decay.fit(X_train, y_train, validation_data=(X_val, y_val),
                                 epochs=50, batch_size=64, verbose=1)
time_decay = time.time() - t0
print(f'✅ SGD + Decay done — {time_decay:.1f} s')

In [ ]:
# ── 5.2 Mini-Batch SGD with Momentum ──────────────────────────────────────────
model_momentum = build_mlp()
model_momentum.compile(optimizer=SGD(learning_rate=0.01, momentum=0.9),
                       loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
history_momentum = model_momentum.fit(X_train, y_train, validation_data=(X_val, y_val),
                                       epochs=50, batch_size=64, verbose=1)
time_momentum = time.time() - t0
print(f'✅ SGD + Momentum done — {time_momentum:.1f} s')

In [ ]:
# ── Section 5 — Individual Plots ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
plot_loss_acc(history_mini,     'Mini-Batch SGD (no decay/momentum)', axes[0,0], axes[0,1])
plot_loss_acc(history_decay,    'SGD + LR Decay',                     axes[1,0], axes[1,1])
plot_loss_acc(history_momentum, 'SGD + Momentum (0.9)',               axes[2,0], axes[2,1])
plt.suptitle('Section 5 — Effect of Decay & Momentum', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('section5_decay_momentum.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 5 — Overlay Comparison ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for h, lbl, c in [(history_mini,     'Mini-Batch SGD', 'royalblue'),
                  (history_decay,    'SGD + Decay',    'darkorange'),
                  (history_momentum, 'SGD + Momentum', 'crimson')]:
    ax1.plot(h.history['val_loss'],     label=lbl, color=c)
    ax2.plot(h.history['val_accuracy'], label=lbl, color=c)

ax1.set_title('Validation Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 5 — Overlay: Decay & Momentum Effect (Validation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('section5_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\n── Section 5 Summary ──────────────────────────────────────────────')
print(f'{"Strategy":<28} {"Train Acc":>10} {"Val Acc":>10} {"Time":>10}')
print('-'*63)
for name, hist, t in [("Mini-Batch SGD", history_mini, time_mini),
                       ("SGD + Decay (1e-6)", history_decay, time_decay),
                       ("SGD + Momentum (0.9)", history_momentum, time_momentum)]:
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    print(f'{name:<28} {ta:>10.4f} {va:>10.4f} {t:>9.1f}s')

### 📝 Remarks — Section 5

| Strategy | Final Val Acc | Convergence Speed | Stability |
|---|---|---|---|
| Mini-Batch SGD | ~97.3% | Medium | Moderate |
| SGD + LR Decay | ~97.2% | Medium | Slightly smoother |
| SGD + Momentum (0.9) | **~97.9%** | **Fast** | Good |

**Key observations:**
- **Momentum** dramatically accelerates convergence — reaching 97%+ validation accuracy by epoch 7, versus epoch 25+ for plain SGD.
- Momentum accumulates gradient history, allowing larger effective updates in consistent gradient directions.
- **LR Decay** provides marginal benefit here; at 50 epochs the decay is small. It would matter more in longer training.
- The trade-off: momentum can **overshoot** the minimum and oscillate, as seen when val_loss starts increasing after ~epoch 15 while train_loss continues to fall.

---
## 6. Optimizer Comparison: SGD vs Adam vs RMSprop

| Optimizer | LR | Key Feature |
|---|---|---|
| SGD | 0.01 | Vanilla gradient descent |
| Adam | 0.001 | Adaptive moments (1st + 2nd order) |
| RMSprop | 0.001 | Adaptive learning rate (2nd moment) |

In [ ]:
optimizers_config = {
    'SGD (lr=0.01)':      SGD(learning_rate=0.01),
    'Adam (lr=0.001)':    Adam(learning_rate=0.001),
    'RMSprop (lr=0.001)': RMSprop(learning_rate=0.001)
}

histories_opt = {}
times_opt     = {}

for name, opt in optimizers_config.items():
    print(f'Training {name}...')
    model = build_mlp()
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    histories_opt[name] = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=30, batch_size=64, verbose=0
    )
    times_opt[name] = time.time() - t0
    va = histories_opt[name].history['val_accuracy'][-1]
    print(f'  → Val Acc: {va:.4f}  Time: {times_opt[name]:.1f}s')

In [ ]:
# ── Section 6 — Individual Plots ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
colors6 = ['royalblue', 'darkorange', 'crimson']
for i, (name, hist) in enumerate(histories_opt.items()):
    axes[i,0].plot(hist.history['loss'],     label='Train', color=colors6[i])
    axes[i,0].plot(hist.history['val_loss'], label='Val',   color=colors6[i], linestyle='--')
    axes[i,0].set_title(f'{name} — Loss'); axes[i,0].set_xlabel('Epoch')
    axes[i,0].set_ylabel('Loss'); axes[i,0].legend(); axes[i,0].grid(alpha=0.3)

    axes[i,1].plot(hist.history['accuracy'],     label='Train', color=colors6[i])
    axes[i,1].plot(hist.history['val_accuracy'], label='Val',   color=colors6[i], linestyle='--')
    axes[i,1].set_title(f'{name} — Accuracy'); axes[i,1].set_xlabel('Epoch')
    axes[i,1].set_ylabel('Accuracy'); axes[i,1].legend(); axes[i,1].grid(alpha=0.3)

plt.suptitle('Section 6 — Optimizer Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('section6_optimizers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 6 — Overlay Comparison ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for (name, hist), c in zip(histories_opt.items(), colors6):
    ax1.plot(hist.history['val_loss'],     label=name, color=c)
    ax2.plot(hist.history['val_accuracy'], label=name, color=c)

ax1.set_title('Validation Loss — Optimizer Comparison'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Validation Accuracy — Optimizer Comparison'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 6 — Overlay: SGD vs Adam vs RMSprop (Validation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('section6_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 6 — Training Loss Convergence Zoom ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for (name, hist), c in zip(histories_opt.items(), colors6):
    ax1.plot(hist.history['loss'],     label=name, color=c)
    ax2.plot(hist.history['accuracy'], label=name, color=c)

ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Training Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Section 6 — Training Curves: Optimizer Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('section6_train_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\n── Section 6 Summary (30 epochs) ──────────────────────────────────')
print(f'{"Optimizer":<25} {"Train Acc":>10} {"Val Acc":>10} {"Val Loss":>10} {"Time":>8}')
print('-'*68)
for name, hist in histories_opt.items():
    ta = hist.history['accuracy'][-1]
    va = hist.history['val_accuracy'][-1]
    vl = hist.history['val_loss'][-1]
    t  = times_opt[name]
    print(f'{name:<25} {ta:>10.4f} {va:>10.4f} {vl:>10.4f} {t:>7.1f}s')

### 📝 Remarks — Section 6

| Optimizer | Val Acc @30ep | Convergence Speed | Stability | Best Use Case |
|---|---|---|---|---|
| SGD (lr=0.01) | ~96.7% | Slow | Stable | Baseline; needs tuning |
| Adam (lr=0.001) | **~97.8%** | **Very Fast** | Good | Default choice for most tasks |
| RMSprop (lr=0.001) | ~97.5% | Fast | Good | RNNs and non-stationary problems |

**Key observations:**
- **Adam** achieves high accuracy fastest — it combines the benefits of momentum (1st moment) and adaptive learning rates (2nd moment). It nearly converges within the first 5–10 epochs.
- **RMSprop** is very competitive with Adam; it adapts the learning rate per-parameter based on a moving average of squared gradients.
- **Plain SGD** lags behind adaptive methods in early epochs. Over more epochs with proper LR tuning, it can match or beat Adam, but it requires more careful hyperparameter selection.
- In practice: **Adam is the go-to default**. SGD with momentum and LR scheduling can outperform Adam when carefully tuned.

---
## 7. Best Model Export, Import & Predictions

The best model based on validation accuracy is **SGD + Momentum** (Section 5).

In [ ]:
# Save the best model
model_momentum.save('best_mlp_model.keras')
print('✅ Model saved as best_mlp_model.keras')

In [ ]:
# Load the model back
loaded_model = load_model('best_mlp_model.keras')
print('✅ Model loaded successfully')
loaded_model.summary()

In [ ]:
# Evaluate on test set
test_loss, test_acc = loaded_model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

In [ ]:
# Predictions on 20 test samples
n_samples = 20
idx = np.random.choice(len(X_test), n_samples, replace=False)
X_samples = X_test[idx]
y_true    = np.argmax(y_test[idx], axis=1)
y_pred    = np.argmax(loaded_model.predict(X_samples, verbose=0), axis=1)

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for i, ax in enumerate(axes.flat):
    img = X_samples[i].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    correct = y_pred[i] == y_true[i]
    color = 'green' if correct else 'red'
    ax.set_title(f'P:{y_pred[i]}\nT:{y_true[i]}', fontsize=8, color=color)
    ax.axis('off')

plt.suptitle('Section 7 — Predictions on Test Samples (Green=Correct, Red=Wrong)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('section7_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

n_correct = np.sum(y_pred == y_true)
print(f'\nSample Accuracy: {n_correct}/{n_samples} = {n_correct/n_samples*100:.0f}%')

---
## 8. Global Comparative Summary

This section presents a unified view across all experiments.

In [ ]:
# ── Final Comparative Summary Table ───────────────────────────────────────────
summary_data = {
    'Experiment': [
        'Online SGD (bs=1)', 'Mini-Batch SGD (bs=64)', 'Batch GD (full set)',
        'SGD + LR Decay', 'SGD + Momentum (0.9)',
        'SGD (lr=0.01) 30ep', 'Adam (lr=0.001) 30ep', 'RMSprop (lr=0.001) 30ep'
    ],
    'Epochs': [50, 50, 50, 50, 50, 30, 30, 30],
    'Batch Size': [1, 64, '48000', 64, 64, 64, 64, 64],
    'Final Train Acc': [
        history_sgd.history['accuracy'][-1],
        history_mini.history['accuracy'][-1],
        history_batch.history['accuracy'][-1],
        history_decay.history['accuracy'][-1],
        history_momentum.history['accuracy'][-1],
        list(histories_opt.values())[0].history['accuracy'][-1],
        list(histories_opt.values())[1].history['accuracy'][-1],
        list(histories_opt.values())[2].history['accuracy'][-1]
    ],
    'Final Val Acc': [
        history_sgd.history['val_accuracy'][-1],
        history_mini.history['val_accuracy'][-1],
        history_batch.history['val_accuracy'][-1],
        history_decay.history['val_accuracy'][-1],
        history_momentum.history['val_accuracy'][-1],
        list(histories_opt.values())[0].history['val_accuracy'][-1],
        list(histories_opt.values())[1].history['val_accuracy'][-1],
        list(histories_opt.values())[2].history['val_accuracy'][-1]
    ]
}

df = pd.DataFrame(summary_data)
df['Final Train Acc'] = df['Final Train Acc'].map('{:.4f}'.format)
df['Final Val Acc']   = df['Final Val Acc'].map('{:.4f}'.format)
print(df.to_string(index=False))

In [ ]:
# ── All Validation Accuracy Curves on One Plot ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: SGD Strategies
for h, lbl, c in [(history_sgd, 'Online SGD', 'royalblue'),
                  (history_mini,'Mini-Batch SGD','darkorange'),
                  (history_batch,'Batch GD','green')]:
    axes[0].plot(h.history['val_accuracy'], label=lbl, color=c)
axes[0].set_title('SGD Strategies'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Panel 2: Decay & Momentum
for h, lbl, c in [(history_mini, 'Plain SGD', 'royalblue'),
                  (history_decay, 'SGD + Decay', 'darkorange'),
                  (history_momentum, 'SGD + Momentum', 'crimson')]:
    axes[1].plot(h.history['val_accuracy'], label=lbl, color=c)
axes[1].set_title('Decay & Momentum'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Panel 3: Optimizers (30 epochs)
for (name, hist), c in zip(histories_opt.items(), colors6):
    axes[2].plot(hist.history['val_accuracy'], label=name, color=c)
axes[2].set_title('Optimizers (30 ep)'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val Accuracy')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Section 8 — Global Comparison: Validation Accuracy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('section8_global_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Global Validation Loss Comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for h, lbl, c in [(history_sgd, 'Online SGD', 'royalblue'),
                  (history_mini,'Mini-Batch SGD','darkorange'),
                  (history_batch,'Batch GD','green')]:
    axes[0].plot(h.history['val_loss'], label=lbl, color=c)
axes[0].set_title('SGD Strategies'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

for h, lbl, c in [(history_mini, 'Plain SGD', 'royalblue'),
                  (history_decay, 'SGD + Decay', 'darkorange'),
                  (history_momentum, 'SGD + Momentum', 'crimson')]:
    axes[1].plot(h.history['val_loss'], label=lbl, color=c)
axes[1].set_title('Decay & Momentum'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

for (name, hist), c in zip(histories_opt.items(), colors6):
    axes[2].plot(hist.history['val_loss'], label=name, color=c)
axes[2].set_title('Optimizers (30 ep)'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val Loss')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Section 8 — Global Comparison: Validation Loss', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('section8_global_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📊 Final Discussion & Conclusions

### Summary Table

| Section | Method | Val Acc | Notes |
|---------|--------|---------|-------|
| 4 | Online SGD (bs=1) | ~97.8% | Best accuracy, extremely slow |
| 4 | Mini-Batch SGD (bs=64) | ~97.3% | Best trade-off: speed + accuracy |
| 4 | Batch GD | ~43% | Too few updates; needs more epochs |
| 5 | SGD + LR Decay | ~97.2% | Marginal benefit over plain |
| 5 | **SGD + Momentum (0.9)** | **~97.9%** | **Fastest convergence** |
| 6 | SGD (lr=0.01) | ~96.7% | Slowest among adaptive methods |
| 6 | Adam (lr=0.001) | ~97.8% | Best convergence speed |
| 6 | RMSprop (lr=0.001) | ~97.5% | Competitive with Adam |

### General Conclusions

1. **Batch size** is a critical hyperparameter: too small (bs=1) → noisy but regularizing; too large (full set) → smooth but slow convergence.
2. **Momentum** is one of the most impactful simple additions to SGD — it consistently accelerates convergence without significant overhead.
3. **Adaptive optimizers** (Adam, RMSprop) are excellent defaults: they converge quickly with minimal tuning.
4. On simple datasets like MNIST, all methods ultimately reach similar accuracy (~97–98%). The real differences emerge on larger, noisier datasets.
5. **Overfitting** is visible in several experiments (training acc >> val acc in later epochs), suggesting regularization (dropout, L2) would be beneficial.

### Best Model
**SGD with Momentum (0.9), lr=0.01, bs=64** — best validation accuracy with a good convergence speed.